# Deriving QA thresholds

The procedure, run top to bottom: measure the [validation visits](validation-visits.md)
with the current code, derive `WARN`/`FAIL` thresholds from the known-good visits, check
them against the known-bad ones, and review them for adoption. Run it whenever the metrics
code, the visit set or the instrument changes. The rules behind it are in
[`qa-principles.md`](qa-principles.md).

Sections 3 and 4 run `pipetask`. Section 4 is the only one that writes to the Butler (one
new collection), and only when `RUN_PIPELINE` is set. Everything else reads.

**Start Jupyter from a shell with the stack set up**, so that `pipetask` and the Butler are
available to the kernel:

```bash
source /path/to/stack/loadLSST.bash
setup -r /path/to/drp_stella
setup -r /path/to/drp_qa
cd /path/to/drp_qa/docs
jupyter lab qa-thresholds.ipynb
```

Set the inputs in section 1, then run all cells. When it has run, commit it with its
outputs (section 9): they are the record of what the
thresholds were derived from.

Confirm the setup; the output is the record of what was run:

In [ ]:
!eups list --setup drp_stella

In [ ]:
!eups list --setup drp_qa

## 1. Inputs

The only cell to edit. Each run writes a numbered collection, `u/$USER/qa-thresholds/001`,
`002`, ..., never one that mixes code versions such as `qaActor/reductions`.
`RUN_PIPELINE = True` makes the next numbered run; `False` reads the latest one again. To
read an older run, set `RUN_NAME` to its collection.

In [ ]:
import os

# Likely to change
RUN_PIPELINE = False  # True: run imageQualityQa into a new numbered collection (section 4)
RUN_NAME = None  # optional: an older run to read, e.g. "u/someone/qa-thresholds/002"

# Rarely change
PREFIX = f"u/{os.environ['USER']}/qa-thresholds"
BUTLER = "/work/datastore"
INPUTS = "PFS/defaults"  # raw data and calibrations; the notebook reduces everything itself
FILES = "~/.cache/drp_qa"  # pipetask logs and the metrics cache

## Set up

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pfs.drp.qa.metrics.calibration import (
    DEFAULT_METRICS,
    METRIC_SPECS,
    calibrate,
    compareRuns,
    labelRows,
    summarizeRuns,
)
from pfs.drp.qa.metrics.readers import chooseRun, readMetrics
from pfs.drp.qa.metrics.validationVisits import (
    loadValidationVisits,
    sequenceTypeMismatches,
    unmatchedEntries,
    visitExpression,
)
from pfs.drp.qa.plotting import plotThresholds

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

existing = !butler query-collections {BUTLER} "{PREFIX}/*"
COLLECTION = chooseRun(existing, PREFIX, newRun=RUN_PIPELINE, runName=RUN_NAME)
PIPELINE = "../pipelines/qaThresholds.yaml"
FILES = Path(FILES).expanduser()
FILES.mkdir(parents=True, exist_ok=True)
STEM = COLLECTION.replace("/", "_")
CACHE = FILES / f"iqQaMetrics-{STEM}.parquet"
METRICS = [*DEFAULT_METRICS, "nLines"]
print(f"collection: {COLLECTION}")
print(f"files:      {FILES}")

## 2. The visits

From the YAML, never typed by hand, in two passes. `drpActor` sets the reduction config by
sequence type (`ics_drpActor`, `engine.newVisitGroup`), and the thresholds must describe the
data as live QA sees it, so each pass uses that sequence type's config.

In [ ]:
visitSet = loadValidationVisits()
print(
    f"{len(visitSet.knownGood)} known_good entries ({len(visitSet.goodVisits)} visits), "
    f"{len(visitSet.confirmedBad)} known_bad, {len(visitSet.unconfirmedBad)} unconfirmed; "
    f"{len(visitSet.visits)} visits"
)

# drpActor's per-sequence-type config overrides.
PASSES = {
    "arcs and traces": (
        visitSet.visitsOfType("scienceArc", "scienceTrace"),
        {
            "reduceExposure:requireAdjustDetectorMap": False,
            "isr:h4.quickCDS": True,
            "cosmicray:doNormalizeChiRms": True,
        },
    ),
    "sky": (
        visitSet.visitsOfType("scienceObject"),
        {
            "reduceExposure:requireAdjustDetectorMap": True,
            "isr:h4.quickCDS": False,
            "cosmicray:doNormalizeChiRms": True,
        },
    ),
}


def passArgs(visits, config):
    """Return the -c options and the -d expression of one pass."""
    options = " ".join(f"-c {key}={value}" for key, value in config.items())
    return options, f"instrument = 'PFS' AND {visitExpression(visits)}"


for name, (visits, config) in PASSES.items():
    options, where = passArgs(visits, config)
    print(f"{name}: {len(visits)} visits; {options}")
    print(f"    {where}")

## 3. Check the inputs exist

Build each pass's graph without running it. The pipeline reduces from raw data (isr,
cosmicray, reduceExposure) and then runs `imageQualityQa`: three quanta per detector, four
on the n arm, where cosmicray runs. For the current set that is about 4,500 quanta over
the 1,328 detectors read (run 002: 3,538 for arcs and traces, 960 for sky). Far fewer means raw data or
calibrations are missing from `INPUTS`. Section 5 lists any visit that ends up with no
metrics.

In [ ]:
for name, (visits, config) in PASSES.items():
    options, where = passArgs(visits, config)
    log = FILES / f"qgraph-{STEM}-{name.replace(' ', '_')}.log"
    print(f"{name}:")
    !pipetask qgraph -b {BUTLER} -p {PIPELINE} -i {INPUTS} -o {COLLECTION} {options} -d "{where}" > {log} 2>&1
    !grep -iE "quanta|error" {log} | tail -n 5

## 4. Reduce and measure

Only with `RUN_PIPELINE = True` (section 1), into the next numbered collection, which the
set-up cell chose (it refuses `RUN_PIPELINE` with `RUN_NAME`, which would write into an
older run). Each pass adds a run to that collection. Running all cells again with
`RUN_PIPELINE` still set makes another numbered run: set it back to `False` to re-read this
one. The full logs go to files; the next cell reports their outcome.

In [ ]:
if RUN_PIPELINE:
    for name, (visits, config) in PASSES.items():
        options, where = passArgs(visits, config)
        log = FILES / f"pipetask-{STEM}-{name.replace(' ', '_')}.log"
        print(f"{name}:")
        !pipetask --long-log --log-level PFS=INFO run -j 8 -b {BUTLER} -p {PIPELINE} -i {INPUTS} -o {COLLECTION} {options} -d "{where}" > {log} 2>&1
        !tail -n 3 {log}
else:
    print(f"RUN_PIPELINE is False: using the existing {COLLECTION}")

The outcome of each pass, read from its log file so that it is recorded here even when
the run finished with the notebook closed: the executed and failed quanta.

In [ ]:
for name in PASSES:
    log = FILES / f"pipetask-{STEM}-{name.replace(' ', '_')}.log"
    print(f"{name}: {log}")
    if not log.exists():
        print("    no log: this run was made elsewhere, or not yet")
        continue
    for line in log.read_text().splitlines():
        if ("Executed" in line and "quanta" in line) or " - FAILED" in line:
            print("    " + line.split(" - ", 1)[-1].strip())

## 5. Read the metrics

Read-only, once; later runs read the cache. Delete the cache file to re-read.

In [ ]:
if CACHE.exists():
    metrics = pd.read_parquet(CACHE)
else:
    from lsst.daf.butler import Butler

    butler = Butler(BUTLER, collections=[COLLECTION], writeable=False)
    metrics = readMetrics(butler, visitSet.visits, collections=[COLLECTION])
    metrics.to_parquet(CACHE)
print(f"{len(metrics)} rows over {metrics['visit'].nunique()} visits")
missing = sorted(set(visitSet.visits) - set(metrics["visit"]))
print(f"{len(missing)} validation visits have no metrics: {missing}")

# obsType comes from the W_SEQTYP header: a disagreement means a wrong
# sequenceType in the YAML, and a visit reduced with the wrong config.
mismatched = sequenceTypeMismatches(metrics, visitSet)
for (visit, obsType, sequenceType), _ in mismatched.groupby(["visit", "obsType", "sequenceType"]):
    print(f"visit {visit}: obsType {obsType!r} but sequenceType {sequenceType!r} in the YAML")

# An entry that matches no row is a check that silently does not run: its
# visits were not processed, or a selector (often seqType) is wrong.
for entry in unmatchedEntries(metrics, visitSet):
    print(
        f"matches no rows: visits {entry.visits[0]}-{entry.visits[-1]} "
        f"seqType={entry.seqType!r} expect={entry.expect}"
    )

## 6. Suggested thresholds

Derived from the known-good visits of the reference run (Run25) only. One row per metric
and population: arm and observation type; flag rates also per lamp; line counts per lamp.
The known-good visits of other runs are held out: `heldOutFlaggedWarn` and
`heldOutFlaggedFail` are how often the thresholds flag them.

In [ ]:
table = calibrate(metrics, visitSet, METRICS)
columns = [
    "metric",
    "group",
    "nGood",
    "nGoodVisits",
    "median",
    "robustRms",
    "warn",
    "fail",
    "failLow",
    "failHigh",
    "goodFlaggedWarn",
    "goodFlaggedFail",
    "reliable",
    "failBounded",
    "degenerate",
    "nBad",
    "badOk",
    "nUnconfirmed",
    "unconfirmedFlagged",
]
table[columns].round(4)

In [ ]:
for _, row in table[table["nBad"] > 0].iterrows():
    print(f"{row['metric']}[{row['group']}]: {row['badSummary']}")

### Other runs against the reference

Each run's known-good data against the reference thresholds. The reference run's rows are
in-sample, near 5 % and 1 % by construction; a held-out run flagged much more often has
moved, or its data differ. Run27 had no calibrations of its own, so expect it to stand out.

In [ ]:
compareRuns(metrics, visitSet, table, METRICS).round(3)

### Reported, not derived: `medDxCenter`

An offset from `detectorMap_calib` is near zero in the run the calibrations were made from,
so percentiles of Run25 say nothing about how much drift to tolerate: its thresholds are a
tolerance, set in PIPE2D-1921. Per run, of `|medDxCenter|` in px:

In [ ]:
summarizeRuns(metrics, visitSet, ["medDxCenter"]).round(3)

Per population: the CDF of the reference run's known-good values, `WARN` (amber) and `FAIL` (red) with
the 95 % interval on the FAIL percentile shaded, and below it the other runs' known-good
values (blue dots), the known-bad values (crosses expect FAIL, triangles WARN) and the
unconfirmed ones (open circles). A known-bad marker left
of FAIL (right of it, for `nLines`) is a fault the threshold misses.

In [ ]:
for metric in METRICS:
    labelled = labelRows(metrics, visitSet, metric)
    plotThresholds(labelled, table, metric, absolute=METRIC_SPECS[metric].absolute)
    plt.show()

The provenance sentences, for the `doc` of each threshold adopted:

In [ ]:
for _, row in table[table["nGood"] > 0].iterrows():
    print(f"{row['metric']}[{row['group']}]: warn={row['warn']:g} fail={row['fail']:g}")
    print(f"    {row['provenance']}")

## 7. Review

Go through these in order:

1. **Missing visits, sequence types and unmatched entries** (section 5). A missing visit or
   an unmatched entry is a check that didn't run; a sequence-type disagreement means a
   visit was reduced with the wrong config.
   An entry whose visits are present but which matches nothing has a wrong selector:
   compare its `seqType` with the `seqName` column, character for character.
2. **Known-bad checks.** In each population with known-bad rows, `badOk` must be true. If
   not, look at the panel: a marker on the good side of `FAIL` is a missed fault. Either the
   metric doesn't see this fault, or the visit isn't bad for the reason recorded. Fix
   whichever is wrong, not the threshold.
3. **Sample size.** `reliable` is false below 20 values. `nGoodVisits` is the more honest
   count: the detectors of one visit, and back-to-back exposures, are correlated.
4. **`failBounded`.** When false, the sample is too small to bound the FAIL percentile
   (about 370 values are needed for p99), so `FAIL` sits at the sample's extreme. Usable,
   but expect it to move as data accumulate; the provenance sentence says so.
5. **`degenerate`.** `WARN` is not below `FAIL`, so `WARN` can never fire. It comes from
   ties in the tail; set the pair by hand from the plot.
6. **Held-out runs.** `heldOutFlaggedFail` and the per-run table show how the thresholds
   do on data they were not derived from. A run flagged far more than the reference is a
   change to explain (calibrations, focus, a pipeline version), not a threshold to widen.
7. **In-sample flag rates.** `goodFlaggedWarn` and `goodFlaggedFail` should be near 5 % and
   1 %. Much more means ties at the threshold.
8. **Unconfirmed visits.** `unconfirmedFlagged` counts the suspected faults the suggestion
   would flag. Use it to settle them in the YAML ([`validation-visits.md`](validation-visits.md)),
   not to move a threshold.
9. **Populations without good data.** Rows with `nGood` 0 have known-bad visits and nothing
   to compare them with. Add known-good visits for that population, or accept that it goes
   unchecked.

## 8. Adopt

The thresholds are config fields of `ImageQualityQaConfig` in
`python/pfs/drp/qa/imageQualityQa.py`. Changing them changes verdicts, so it is done on a
ticket of its own (for the first derivation, PIPE2D-1917).

| Metric | Fields | Granularity | Populations it gates |
|---|---|---|---|
| `medFwhm` | `fwhmWarnThreshold`, `fwhmFailThreshold` | one value | every arm, except traces (not gated until PIPE2D-1917) |
| `medDxCenter` (absolute) | `dxCenterWarnThreshold`, `dxCenterFailThreshold` | one value | every arm and type |
| `pctFlagged` | `flagRateWarnThreshold`, `flagRateFailThreshold` | per `arm` or `arm:species` | as keyed |

1. Where a field holds one value but this notebook gives one per population, use the most
   lenient suggestion among the populations that field gates. Anything stricter fails good
   detectors in the widest population. Note the populations in the `doc`. The spread
   between populations is what a single value costs: a fault that reaches only the lenient
   population's threshold goes unflagged in the others.
2. Paste the provenance sentence (section 6) into the field's `doc`.
3. Update the threshold tables in the README (`imageQualityQa` → *Pass/Warn/Fail
   Thresholds*) and add a line under `## [Unreleased]` in `CHANGELOG.md`.
4. Run this notebook again with `RUN_PIPELINE = True` (a new numbered run) and the new defaults, and check that every
   known-good visit passes and every known-bad one gets its expected verdict.

## 9. Commit

Commit this notebook with its outputs on the ticket branch. It shows only validation visits,
which are engineering and calibration visits, so its outputs are fine to publish.

```bash
git add docs/qa-thresholds.ipynb
git commit
```